# FranchiseOps AI — Milestone 1: Authentication Module

**Infosys Springboard Internship 7.0 · Batch 4**  
**Stack:** Streamlit + SQLite + bcrypt + JWT + Gmail OTP + ngrok  

This notebook creates an authentication starter for the **FranchiseOps AI** project. It includes signup, login, a JWT-protected user dashboard, security-question recovery, email OTP recovery, and a separate admin console.

## Run order

1. Read the secret setup carefully.
2. Run the dependency cell.
3. Mount Google Drive and set the project folder.
4. Add the six values in **Colab → Secrets** and run the secret-loading cell.
5. Run each `%%writefile` cell to create `database.py`, `auth_utils.py`, `app.py`, `requirements.txt`, and `README.md`.
6. Run the database/syntax checks.
7. Run the launch cell and open the ngrok URL.
8. Test the flows and run the stop cell when finished.

**Security:** This is a student demo, not a production authentication service. Use test data only. Anyone with the public ngrok URL may reach the login page. Stop the tunnel after the demo.


## Before running: configure Colab Secrets

In Google Colab, click the **key icon (Secrets)** in the left sidebar. Add each value below and turn **Notebook access** on for every secret.

| Secret name | What to put in it |
|---|---|
| `JWT_SECRET` | A unique random signing secret, at least 32 characters. Do not reuse a normal account password. |
| `NGROK_AUTHTOKEN` | The authtoken from your [ngrok dashboard](https://dashboard.ngrok.com/get-started/your-authtoken). |
| `EMAIL_ADDRESS` | The Gmail address that will send password-reset OTPs. |
| `EMAIL_PASSWORD` | A Gmail **App Password**, not your regular Google password. |
| `ADMIN_USERNAME` | Your private admin username. |
| `ADMIN_PASSWORD` | Your strong admin password. |

**Important:** Your brief says the admin credentials should be in code, but Step 12 also says to remove hard-coded admin passwords before upload. This notebook resolves that conflict safely by keeping admin credentials in Colab Secrets. Do not add real credentials to a code cell or GitHub. Gmail App Passwords require 2-Step Verification; if Google does not offer App Passwords for your account, account/organisation policy may restrict them.


## Step 1 — Install the required packages

Run this first. Colab may ask you to restart the runtime only if a package upgrade specifically requires it; if prompted, reconnect and continue from the Drive/Secrets cells.


In [ ]:
%pip install -q "streamlit>=1.40,<2.0" "pyngrok>=7.2,<9.0" "PyJWT>=2.8,<3.0" "bcrypt>=4.1,<6.0"


## Step 2 — Mount Google Drive and create the Milestone1 workspace

The database is saved in this folder so it normally survives a Colab runtime reset. Do not upload the database file to GitHub.


In [ ]:
from pathlib import Path
import os
from google.colab import drive

drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/FranchiseOpsAI/Milestone1')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / 'screenshots').mkdir(exist_ok=True)
os.chdir(PROJECT_DIR)
os.environ['PROJECT_DIR'] = str(PROJECT_DIR)
print('Project folder created:', PROJECT_DIR)
print('Current working directory:', Path.cwd())


## Step 3 — Load secrets without printing them

Run this only after adding all six secrets and enabling notebook access. The values are copied to the notebook runtime environment so the Streamlit subprocess can read them; they are never printed by this cell.


In [ ]:
import os
from google.colab import userdata

REQUIRED_SECRETS = [
    'JWT_SECRET',
    'NGROK_AUTHTOKEN',
    'EMAIL_ADDRESS',
    'EMAIL_PASSWORD',
    'ADMIN_USERNAME',
    'ADMIN_PASSWORD',
]
missing_secrets = []

for secret_name in REQUIRED_SECRETS:
    try:
        secret_value = userdata.get(secret_name)
    except Exception:
        secret_value = None
    if not secret_value or not str(secret_value).strip():
        missing_secrets.append(secret_name)
    else:
        # Preserve passwords exactly; trim accidental spaces from ordinary settings.
        if secret_name in {'EMAIL_PASSWORD', 'ADMIN_PASSWORD'}:
            os.environ[secret_name] = str(secret_value)
        else:
            os.environ[secret_name] = str(secret_value).strip()

if missing_secrets:
    raise RuntimeError(
        'Missing Colab Secrets: ' + ', '.join(missing_secrets) +
        '. Add them in the Secrets panel, enable notebook access, then rerun this cell.'
    )

if len(os.environ['JWT_SECRET']) < 32:
    raise ValueError('JWT_SECRET must be at least 32 characters. Replace it in Colab Secrets.')

os.environ['FRANCHISEOPS_DB_PATH'] = str(PROJECT_DIR / 'franchiseops_users.db')
print('All required secret names were found. No secret values were displayed.')
print('Database path:', os.environ['FRANCHISEOPS_DB_PATH'])


## Step 4 — Database module (`database.py`)

This module creates the users table and uses parameterized SQL queries. It never returns password hashes to the admin dashboard.


In [ ]:
%%writefile database.py
"""SQLite database helpers for FranchiseOps AI Milestone 1."""
import os
import sqlite3
from contextlib import contextmanager

DB_PATH = os.environ.get(
    "FRANCHISEOPS_DB_PATH",
    os.path.join(os.path.dirname(os.path.abspath(__file__)), "franchiseops_users.db"),
)


@contextmanager
def db_connection():
    """Open a database connection and commit or roll back safely."""
    os.makedirs(os.path.dirname(os.path.abspath(DB_PATH)), exist_ok=True)
    conn = sqlite3.connect(DB_PATH, timeout=30)
    conn.row_factory = sqlite3.Row
    try:
        conn.execute("PRAGMA foreign_keys = ON")
        yield conn
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()


def init_db():
    """Create the user table if it does not already exist."""
    with db_connection() as conn:
        conn.executescript(
            """
            CREATE TABLE IF NOT EXISTS users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT NOT NULL COLLATE NOCASE UNIQUE,
                email TEXT NOT NULL COLLATE NOCASE UNIQUE,
                password_hash TEXT NOT NULL,
                security_question TEXT NOT NULL,
                security_answer_hash TEXT NOT NULL,
                created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP,
                reset_otp_hash TEXT,
                reset_otp_expires_at REAL,
                reset_otp_attempts INTEGER NOT NULL DEFAULT 0,
                reset_otp_last_sent_at REAL
            );
            """
        )


def _as_dict(row):
    return dict(row) if row is not None else None


def get_user_by_id(user_id):
    with db_connection() as conn:
        row = conn.execute("SELECT * FROM users WHERE id = ?", (user_id,)).fetchone()
    return _as_dict(row)


def get_user_by_username(username):
    with db_connection() as conn:
        row = conn.execute(
            "SELECT * FROM users WHERE username = ? COLLATE NOCASE",
            (username.strip(),),
        ).fetchone()
    return _as_dict(row)


def get_user_by_email(email):
    with db_connection() as conn:
        row = conn.execute(
            "SELECT * FROM users WHERE email = ? COLLATE NOCASE",
            (email.strip(),),
        ).fetchone()
    return _as_dict(row)


def get_user_by_login(username_or_email):
    """Allow login with either the username or email address."""
    value = username_or_email.strip()
    with db_connection() as conn:
        row = conn.execute(
            "SELECT * FROM users WHERE username = ? COLLATE NOCASE OR email = ? COLLATE NOCASE",
            (value, value),
        ).fetchone()
    return _as_dict(row)


def create_user(username, email, password_hash, security_question, security_answer_hash):
    """Insert a new user. Returns (success, reason)."""
    try:
        with db_connection() as conn:
            conn.execute(
                """INSERT INTO users
                   (username, email, password_hash, security_question, security_answer_hash)
                   VALUES (?, ?, ?, ?, ?)""",
                (username.strip(), email.strip().lower(), password_hash,
                 security_question, security_answer_hash),
            )
        return True, "created"
    except sqlite3.IntegrityError as exc:
        # Do not expose internal SQL details to the user.
        message = str(exc).lower()
        if "users.username" in message:
            return False, "username"
        if "users.email" in message:
            return False, "email"
        return False, "duplicate"


def list_users():
    """Return only safe user information for the admin dashboard."""
    with db_connection() as conn:
        rows = conn.execute(
            "SELECT username, email, created_at FROM users ORDER BY created_at DESC"
        ).fetchall()
    return [dict(row) for row in rows]


def save_otp(email, otp_hash, expires_at, sent_at):
    """Save a hashed OTP and its expiry for a registered email."""
    with db_connection() as conn:
        conn.execute(
            """UPDATE users
               SET reset_otp_hash = ?, reset_otp_expires_at = ?,
                   reset_otp_attempts = 0, reset_otp_last_sent_at = ?
               WHERE email = ? COLLATE NOCASE""",
            (otp_hash, expires_at, sent_at, email.strip()),
        )


def increment_otp_attempts(user_id):
    with db_connection() as conn:
        conn.execute(
            "UPDATE users SET reset_otp_attempts = reset_otp_attempts + 1 WHERE id = ?",
            (user_id,),
        )


def clear_otp(user_id):
    with db_connection() as conn:
        conn.execute(
            """UPDATE users SET reset_otp_hash = NULL, reset_otp_expires_at = NULL,
               reset_otp_attempts = 0 WHERE id = ?""",
            (user_id,),
        )


def update_password(user_id, new_password_hash):
    """Update the password and invalidate any outstanding OTP."""
    with db_connection() as conn:
        conn.execute(
            """UPDATE users SET password_hash = ?, reset_otp_hash = NULL,
               reset_otp_expires_at = NULL, reset_otp_attempts = 0
               WHERE id = ?""",
            (new_password_hash, user_id),
        )


## Step 5 — Security utilities (`auth_utils.py`)

This module handles the milestone email/password rules, bcrypt hashes, signed expiring JWTs, keyed OTP hashes, and Gmail SMTP delivery.


In [ ]:
%%writefile auth_utils.py
"""Validation, password hashing, JWT, OTP hashing, and email helpers."""
import hashlib
import hmac
import os
import re
import smtplib
import ssl
import unicodedata
from datetime import datetime, timedelta, timezone
from email.message import EmailMessage

import bcrypt
import jwt

# Deliberately follows the milestone's simplified email rule.
EMAIL_PATTERN = re.compile(r"^[A-Za-z0-9._%+-]+@(?:[A-Za-z0-9-]+\.)+[A-Za-z]{2,}$")


def validate_email(email):
    """Require 2+ letters in the local part, domain name, and final extension."""
    if not email or not EMAIL_PATTERN.fullmatch(email.strip()):
        return False
    local, domain = email.strip().rsplit("@", 1)
    domain_labels = domain.split(".")
    domain_label_before_extension = domain_labels[-2] if len(domain_labels) >= 2 else ""
    local_letters = sum(character.isalpha() for character in local)
    domain_letters = sum(character.isalpha() for character in domain_label_before_extension)
    return local_letters >= 2 and domain_letters >= 2


def password_issues(password):
    """Return a list of password rules that are not satisfied."""
    issues = []
    if len(password) < 8:
        issues.append("At least 8 characters")
    if not re.search(r"[A-Z]", password):
        issues.append("At least one uppercase letter")
    if not re.search(r"[a-z]", password):
        issues.append("At least one lowercase letter")
    if not re.search(r"\d", password):
        issues.append("At least one number")
    if not re.search(r"[^A-Za-z0-9\s]", password):
        issues.append("At least one special symbol")
    return issues


def normalize_answer(answer):
    """Ignore case and surrounding/repeated whitespace in security answers."""
    normalized = unicodedata.normalize("NFKC", answer or "").strip().casefold()
    return " ".join(normalized.split())


def _bcrypt_input(value):
    """Pre-hash to a fixed-length digest so long inputs avoid bcrypt's 72-byte limit."""
    return hashlib.sha256(value.encode("utf-8")).hexdigest().encode("ascii")


def hash_password(password):
    """Hash a password or security answer with SHA-256 pre-hashing plus bcrypt."""
    return bcrypt.hashpw(_bcrypt_input(password), bcrypt.gensalt(rounds=12)).decode("utf-8")


def verify_hash(value, stored_hash):
    """Safely check a value against a bcrypt hash."""
    try:
        return bcrypt.checkpw(_bcrypt_input(value), stored_hash.encode("utf-8"))
    except (ValueError, TypeError):
        return False


def _jwt_secret():
    secret = os.environ.get("JWT_SECRET", "")
    if len(secret) < 32:
        raise RuntimeError("JWT_SECRET is missing or too short. Configure it in Colab Secrets.")
    return secret


def issue_token(subject, role, username, email=None):
    """Create a signed token that expires after two hours."""
    now = datetime.now(timezone.utc)
    payload = {
        "sub": str(subject),
        "role": role,
        "username": username,
        "iat": now,
        "exp": now + timedelta(hours=2),
    }
    if email:
        payload["email"] = email
    return jwt.encode(payload, _jwt_secret(), algorithm="HS256")


def decode_token(token):
    """Decode and validate a JWT. Return None for expired/invalid tokens."""
    if not token:
        return None
    try:
        return jwt.decode(token, _jwt_secret(), algorithms=["HS256"])
    except (jwt.PyJWTError, RuntimeError, TypeError, ValueError):
        return None


def hash_otp(email, otp):
    """Create a keyed hash so the six-digit OTP is not stored as plain text."""
    message = f"{email.strip().lower()}:{otp}".encode("utf-8")
    return hmac.new(_jwt_secret().encode("utf-8"), message, hashlib.sha256).hexdigest()


def verify_otp(email, otp, stored_hash):
    if not stored_hash:
        return False
    expected = hash_otp(email, otp)
    return hmac.compare_digest(expected, stored_hash)


def send_otp_email(recipient, otp):
    """Send an OTP using Gmail SMTP SSL and a Gmail App Password."""
    sender = os.environ.get("EMAIL_ADDRESS", "").strip()
    # Gmail displays App Passwords in groups; removing spaces works either way.
    app_password = os.environ.get("EMAIL_PASSWORD", "").replace(" ", "").strip()
    if not sender or not app_password:
        raise RuntimeError("EMAIL_ADDRESS or EMAIL_PASSWORD is missing in Colab Secrets.")

    message = EmailMessage()
    message["Subject"] = "FranchiseOps AI - Password Reset OTP"
    message["From"] = sender
    message["To"] = recipient
    message.set_content(
        f"Your FranchiseOps AI password-reset OTP is: {otp}\n\n"
        "This code expires in 10 minutes. Do not share it with anyone. "
        "If you did not request a password reset, you can ignore this email."
    )

    tls_context = ssl.create_default_context()
    with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=tls_context, timeout=30) as server:
        server.login(sender, app_password)
        server.send_message(message)


## Step 6 — Streamlit app (`app.py`)

This is the interactive user interface. Sidebar buttons switch between Login, Sign Up, Forgot Password, and Admin Login. A user dashboard is shown only when a valid user JWT is present.


In [ ]:
%%writefile app.py
"""Streamlit authentication module for FranchiseOps AI - Infosys Springboard Milestone 1."""
import hmac
import os
import re
import secrets
import time

import streamlit as st

from auth_utils import (
    decode_token,
    hash_otp,
    hash_password,
    issue_token,
    normalize_answer,
    password_issues,
    send_otp_email,
    validate_email,
    verify_hash,
    verify_otp,
)
from database import (
    clear_otp,
    create_user,
    get_user_by_email,
    get_user_by_id,
    get_user_by_login,
    get_user_by_username,
    increment_otp_attempts,
    init_db,
    list_users,
    save_otp,
    update_password,
)

st.set_page_config(page_title="FranchiseOps AI | Authentication", page_icon="🏬", layout="wide")

APP_REQUIRED_SECRETS = ["JWT_SECRET", "ADMIN_USERNAME", "ADMIN_PASSWORD", "EMAIL_ADDRESS", "EMAIL_PASSWORD"]
missing = [name for name in APP_REQUIRED_SECRETS if not os.environ.get(name)]
if missing:
    st.error("Missing Colab Secrets: " + ", ".join(missing) + ". Re-run the Secrets setup cell in the notebook.")
    st.stop()
if len(os.environ.get("JWT_SECRET", "")) < 32:
    st.error("JWT_SECRET must be a long random string of at least 32 characters.")
    st.stop()

init_db()

SECURITY_QUESTIONS = [
    "What was the name of your first school?",
    "What city were you born in?",
    "What was your childhood nickname?",
    "What was the first name of your best friend?",
]


def set_view(view_name):
    st.session_state["view"] = view_name


def flash(kind, message):
    st.session_state["flash_message"] = (kind, message)


def get_current_user():
    token = st.session_state.get("jwt_token")
    claims = decode_token(token)
    if not claims or claims.get("role") != "user":
        st.session_state.pop("jwt_token", None)
        return None
    try:
        user = get_user_by_id(int(claims.get("sub", "")))
    except (TypeError, ValueError):
        user = None
    if user is None:
        st.session_state.pop("jwt_token", None)
    return user


def get_current_admin():
    claims = decode_token(st.session_state.get("admin_jwt_token"))
    if claims and claims.get("role") == "admin" and claims.get("sub") == "admin":
        return claims
    st.session_state.pop("admin_jwt_token", None)
    return None


def show_password_rules():
    st.caption("Password rules: 8+ characters, uppercase, lowercase, number, and special symbol.")


def show_password_issues(password):
    issues = password_issues(password)
    if issues:
        st.error("Password must include: " + "; ".join(issues) + ".")
    return not issues


if "view" not in st.session_state:
    st.session_state["view"] = "Login"
if "jwt_token" not in st.session_state:
    st.session_state["jwt_token"] = None
if "admin_jwt_token" not in st.session_state:
    st.session_state["admin_jwt_token"] = None
if "recovery_username" not in st.session_state:
    st.session_state["recovery_username"] = None
if "otp_email" not in st.session_state:
    st.session_state["otp_email"] = None

with st.sidebar:
    st.title("🏬 FranchiseOps AI")
    st.caption("Franchise Operations Intelligence Platform")
    st.divider()
    st.button("Login", on_click=set_view, args=("Login",), use_container_width=True)
    st.button("Sign Up", on_click=set_view, args=("Sign Up",), use_container_width=True)
    st.button("Forgot Password", on_click=set_view, args=("Forgot Password",), use_container_width=True)
    st.button("Admin Login", on_click=set_view, args=("Admin Login",), use_container_width=True)
    st.divider()
    st.caption("Infosys Springboard Internship 7.0 · Batch 4 · Milestone 1")

current_user = get_current_user()
current_admin = get_current_admin()
view = st.session_state.get("view", "Login")

# A valid user JWT protects the Dashboard from direct unauthenticated access.
if current_user and view != "Admin Login":
    st.session_state["view"] = "Dashboard"
    view = "Dashboard"
elif not current_user and view == "Dashboard":
    st.session_state["view"] = "Login"
    view = "Login"

pending_flash = st.session_state.pop("flash_message", None)
if pending_flash:
    kind, message = pending_flash
    getattr(st, kind, st.info)(message)


if view == "Login":
    st.title("Welcome back")
    st.subheader("Login to FranchiseOps AI")
    st.write("Monitor franchise operations from one intelligent workspace.")
    with st.form("login_form"):
        login_identifier = st.text_input("Username or Email *")
        login_password = st.text_input("Password *", type="password")
        login_clicked = st.form_submit_button("Login", use_container_width=True)

    if login_clicked:
        if not login_identifier.strip() or not login_password:
            st.error("Please fill in both fields.")
        else:
            user = get_user_by_login(login_identifier)
            if user and verify_hash(login_password, user["password_hash"]):
                st.session_state["jwt_token"] = issue_token(
                    subject=user["id"], role="user", username=user["username"], email=user["email"]
                )
                st.session_state["view"] = "Dashboard"
                st.rerun()
            else:
                # One generic error avoids saying whether the account exists.
                st.error("Invalid username/email or password.")

    st.caption("New to FranchiseOps AI? Select Sign Up in the sidebar.")


elif view == "Sign Up":
    st.title("Create your account")
    st.write("Create a user account to access the FranchiseOps AI workspace.")
    show_password_rules()
    with st.form("signup_form"):
        username = st.text_input("Username *", max_chars=30)
        email = st.text_input("Email *")
        password = st.text_input("Password *", type="password")
        confirm_password = st.text_input("Confirm Password *", type="password")
        security_question = st.selectbox("Security Question *", SECURITY_QUESTIONS)
        security_answer = st.text_input("Security Answer *", type="password")
        signup_clicked = st.form_submit_button("Create Account", use_container_width=True)

    if signup_clicked:
        username = username.strip()
        email = email.strip().lower()
        if not all([username, email, password, confirm_password, security_question, security_answer.strip()]):
            st.error("Please fill in every required field.")
        elif len(username) < 3:
            st.error("Username must contain at least 3 characters.")
        elif not re.fullmatch(r"[A-Za-z0-9_.-]{3,30}", username):
            st.error("Use 3–30 characters for username: letters, numbers, dot, underscore, or hyphen.")
        elif not validate_email(email):
            st.error("Enter a valid email. This milestone requires at least 2 letters before @, 2 in the domain name, and 2 in the final extension.")
        elif password != confirm_password:
            st.error("Password and Confirm Password do not match.")
        elif password_issues(password):
            show_password_issues(password)
        elif get_user_by_username(username):
            st.error("That username already exists. Choose another username.")
        elif get_user_by_email(email):
            st.error("That email is already registered. Use Login or Forgot Password.")
        else:
            success, reason = create_user(
                username=username,
                email=email,
                password_hash=hash_password(password),
                security_question=security_question,
                security_answer_hash=hash_password(normalize_answer(security_answer)),
            )
            if success:
                st.session_state["recovery_username"] = None
                flash("success", "Account created successfully. Please log in.")
                st.session_state["view"] = "Login"
                st.rerun()
            elif reason == "username":
                st.error("That username already exists. Choose another username.")
            elif reason == "email":
                st.error("That email is already registered. Use Login or Forgot Password.")
            else:
                st.error("Username or email is already registered.")


elif view == "Forgot Password":
    st.title("Reset your password")
    st.write("Choose either security-question recovery or email OTP recovery.")
    recovery_method = st.radio(
        "Recovery method",
        ["Security Question", "Email OTP"],
        horizontal=True,
        key="recovery_method",
    )

    if recovery_method == "Security Question":
        st.subheader("Option A — Security question")
        with st.form("security_lookup_form"):
            lookup_username = st.text_input("Username", key="security_lookup_input")
            lookup_clicked = st.form_submit_button("Show Security Question")
        if lookup_clicked:
            st.session_state["recovery_username"] = None
            recovery_user = get_user_by_username(lookup_username) if lookup_username.strip() else None
            if recovery_user:
                st.session_state["recovery_username"] = recovery_user["username"]
                st.rerun()
            else:
                st.error("Account not found. Check the username and try again.")

        saved_username = st.session_state.get("recovery_username")
        recovery_user = get_user_by_username(saved_username) if saved_username else None
        if recovery_user:
            st.info("Security question: " + recovery_user["security_question"])
            show_password_rules()
            with st.form("security_reset_form"):
                submitted_answer = st.text_input("Your Security Answer *", type="password")
                new_password = st.text_input("New Password *", type="password")
                confirm_new_password = st.text_input("Confirm New Password *", type="password")
                reset_clicked = st.form_submit_button("Reset Password", use_container_width=True)
            if reset_clicked:
                if not all([submitted_answer.strip(), new_password, confirm_new_password]):
                    st.error("Please fill in every required field.")
                elif not verify_hash(normalize_answer(submitted_answer), recovery_user["security_answer_hash"]):
                    st.error("Security answer is incorrect.")
                elif new_password != confirm_new_password:
                    st.error("New Password and Confirm New Password do not match.")
                elif not show_password_issues(new_password):
                    pass
                else:
                    update_password(recovery_user["id"], hash_password(new_password))
                    st.session_state["recovery_username"] = None
                    flash("success", "Password updated. Please log in with your new password.")
                    st.session_state["view"] = "Login"
                    st.rerun()

    else:
        st.subheader("Option B — Email OTP")
        st.caption("OTP expires in 10 minutes. A new code can be requested once every 60 seconds.")
        with st.form("send_otp_form"):
            otp_request_email = st.text_input("Registered Email *", key="otp_request_email")
            send_otp_clicked = st.form_submit_button("Send OTP", use_container_width=True)

        if send_otp_clicked:
            email_value = otp_request_email.strip().lower()
            if not validate_email(email_value):
                st.error("Enter a valid email address.")
            else:
                st.session_state["otp_email"] = email_value
                record = get_user_by_email(email_value)
                generic_notice = "If this email is registered and can receive a code, an OTP has been sent. Check your inbox and spam folder."
                delivery_failed = False
                if record:
                    now = time.time()
                    last_sent = record.get("reset_otp_last_sent_at")
                    if last_sent is None or now - float(last_sent) >= 60:
                        otp_value = f"{secrets.randbelow(1_000_000):06d}"
                        save_otp(email_value, hash_otp(email_value, otp_value), now + 600, now)
                        try:
                            send_otp_email(email_value, otp_value)
                        except Exception:
                            clear_otp(record["id"])
                            delivery_failed = True
                if delivery_failed:
                    st.session_state["otp_send_notice"] = (
                        "The OTP email could not be sent. Check the Gmail App Password and EMAIL_ADDRESS in Colab Secrets, then try again."
                    )
                else:
                    st.session_state["otp_send_notice"] = generic_notice

        notice = st.session_state.pop("otp_send_notice", None)
        if notice:
            st.info(notice)

        target_email = st.session_state.get("otp_email")
        if target_email:
            show_password_rules()
            with st.form("verify_otp_form"):
                otp_input = st.text_input("6-digit OTP *", max_chars=6)
                otp_new_password = st.text_input("New Password *", type="password")
                otp_confirm_password = st.text_input("Confirm New Password *", type="password")
                verify_clicked = st.form_submit_button("Verify OTP and Reset Password", use_container_width=True)

            if verify_clicked:
                record = get_user_by_email(target_email)
                if not record or not record.get("reset_otp_hash"):
                    st.error("OTP is invalid or expired. Request a new code and try again.")
                elif int(record.get("reset_otp_attempts", 0)) >= 5:
                    clear_otp(record["id"])
                    st.error("Too many attempts. Request a new OTP.")
                elif not record.get("reset_otp_expires_at") or time.time() > float(record["reset_otp_expires_at"]):
                    clear_otp(record["id"])
                    st.error("OTP has expired. Request a new code.")
                elif not re.fullmatch(r"\d{6}", otp_input.strip()) or not verify_otp(
                    target_email, otp_input.strip(), record["reset_otp_hash"]
                ):
                    increment_otp_attempts(record["id"])
                    if int(record.get("reset_otp_attempts", 0)) + 1 >= 5:
                        clear_otp(record["id"])
                        st.error("Too many incorrect attempts. Request a new OTP.")
                    else:
                        st.error("OTP is incorrect. Please check the code and try again.")
                elif not all([otp_new_password, otp_confirm_password]):
                    st.error("Please enter and confirm the new password.")
                elif otp_new_password != otp_confirm_password:
                    st.error("New Password and Confirm New Password do not match.")
                elif not show_password_issues(otp_new_password):
                    pass
                else:
                    update_password(record["id"], hash_password(otp_new_password))
                    st.session_state["otp_email"] = None
                    flash("success", "Password updated. Please log in with your new password.")
                    st.session_state["view"] = "Login"
                    st.rerun()


elif view == "Dashboard":
    if not current_user:
        st.error("Please log in to access the dashboard.")
        st.session_state["view"] = "Login"
        st.rerun()
    st.title("FranchiseOps AI Dashboard")
    st.success(f"Welcome, {current_user['username']}!")
    st.write(f"Signed in as **{current_user['email']}**")
    st.info(
        "Milestone 1 is focused on secure authentication. The franchise intelligence modules "
        "will be connected in later milestones; this page does not display fabricated business metrics."
    )
    st.subheader("Project modules")
    module_cols = st.columns(3)
    modules = [
        ("Outlet Performance", "Planned"),
        ("Inventory Optimization", "Planned"),
        ("Workforce Management", "Planned"),
        ("Marketing Intelligence", "Planned"),
        ("Compliance Audit (RAG)", "Planned"),
        ("Customer Feedback", "Planned"),
        ("Weekly Digest & Alerts", "Planned"),
    ]
    for index, (module_name, status) in enumerate(modules):
        with module_cols[index % 3]:
            st.markdown(f"**{module_name}**")
            st.caption(status)
    st.divider()
    if st.button("Logout", type="primary"):
        st.session_state["jwt_token"] = None
        st.session_state["view"] = "Login"
        flash("success", "You have been logged out.")
        st.rerun()


elif view == "Admin Login":
    st.title("Admin Console")
    st.write("This separate admin login is not created through user signup.")
    if current_admin:
        st.success("Admin authentication successful.")
        st.subheader("Registered users")
        users = list_users()
        if users:
            st.dataframe(users, use_container_width=True, hide_index=True)
        else:
            st.info("No registered users yet.")
        st.caption("For privacy, passwords, password hashes, and security answers are never displayed.")
        if st.button("Admin Logout", type="primary"):
            st.session_state["admin_jwt_token"] = None
            st.session_state["view"] = "Login"
            flash("success", "Admin session ended.")
            st.rerun()
    else:
        with st.form("admin_login_form"):
            admin_username = st.text_input("Admin Username *")
            admin_password = st.text_input("Admin Password *", type="password")
            admin_login_clicked = st.form_submit_button("Admin Login", use_container_width=True)
        if admin_login_clicked:
            configured_username = os.environ.get("ADMIN_USERNAME", "")
            configured_password = os.environ.get("ADMIN_PASSWORD", "")
            username_ok = hmac.compare_digest(admin_username, configured_username)
            password_ok = hmac.compare_digest(admin_password, configured_password)
            if username_ok and password_ok:
                st.session_state["admin_jwt_token"] = issue_token(
                    subject="admin", role="admin", username=configured_username
                )
                st.rerun()
            else:
                st.error("Invalid admin username or password.")

st.caption("Educational internship prototype — do not use real customer/employee records in this public demo.")


## Step 7 — Save the dependency list (`requirements.txt`)


In [ ]:
%%writefile requirements.txt
streamlit>=1.40,<2.0
pyngrok>=7.2,<9.0
PyJWT>=2.8,<3.0
bcrypt>=4.1,<6.0


## Step 8 — Write the repository README


In [ ]:
%%writefile README.md
# FranchiseOps AI — Milestone 1: User Authentication

**Infosys Springboard Internship 7.0 · Batch 4**  
**Project:** FranchiseOps AI — Agentic Franchise Operations Intelligence Platform

## 1. What this milestone does

Milestone 1 builds the secure account-access foundation for FranchiseOps AI. It includes user signup, login using username or email, signed JWT sessions, logout, two password-recovery methods (security question and email OTP), and a separate admin console that lists registered usernames and email addresses.

The first user dashboard is intentionally a project shell. It does not invent sales, inventory, staff, or customer metrics; those modules are planned for later milestones.

## 2. Features

- Signup with unique username/email, validated email, password confirmation, security question, and security answer.
- Passwords and security answers stored as bcrypt hashes.
- Login with a generic error message on failed credentials.
- Signed HS256 JWT with a two-hour expiration stored in Streamlit session state.
- Forgot password using the saved security question or a six-digit email OTP.
- OTP is stored as a keyed hash, expires after 10 minutes, allows up to five attempts, and has a 60-second resend cooldown.
- Separate admin login; registered user list never displays password or hash data.
- SQLite database; the Colab notebook stores it in a dedicated Google Drive project folder to persist across ordinary runtime resets.

## 3. Technology stack

Python, Streamlit, SQLite, bcrypt, PyJWT, Gmail SMTP SSL, Google Colab Secrets, Google Drive, and pyngrok.

## 4. Google Colab setup

1. Open `FranchiseOpsAI_Milestone1_Auth.ipynb` in Google Colab and run the notebook cells from top to bottom.
2. In Colab, click **Secrets** (key icon) and add these secrets. Enable notebook access for every one:
   - `JWT_SECRET` — a unique random string of at least 32 characters.
   - `NGROK_AUTHTOKEN` — your ngrok authtoken.
   - `EMAIL_ADDRESS` — the Gmail account that sends OTPs.
   - `EMAIL_PASSWORD` — a Gmail App Password, not your normal Google password.
   - `ADMIN_USERNAME` — a private admin username.
   - `ADMIN_PASSWORD` — a strong private admin password.
3. Run the dependency installation, Google Drive mount, and secret-loading cells.
4. Run the cells that write `database.py`, `auth_utils.py`, and `app.py`.
5. Run the database check, then the launch cell. Open the ngrok public URL printed at the end.
6. Test signup, login, both password recovery methods, user logout, and admin login.
7. When finished, run the stop cell to stop Streamlit and close the ngrok tunnel.

### Gmail App Password notes

Gmail App Passwords require 2-Step Verification. Create one in Google Account settings; store it only in Colab Secrets. Do not paste the app password into a notebook cell or commit it. If the App Password screen is unavailable, your account's security or organisation policy may restrict this feature.

## 5. Local files created by the notebook

```text
Milestone1/
├── FranchiseOpsAI_Milestone1_Auth.ipynb
├── README.md
├── app.py
├── auth_utils.py
├── database.py
├── requirements.txt
├── franchiseops_users.db       # private runtime data; do not upload
├── streamlit.log               # runtime log; do not upload
└── screenshots/
    ├── login.png
    ├── signup.png
    ├── forgot_security_question.png
    ├── forgot_otp.png
    ├── otp_email.png
    ├── user_dashboard.png
    └── admin_dashboard.png
```

## 6. Screenshots

Take screenshots from your running app and save them under `screenshots/` with the filenames below. Do not use made-up screenshots or screenshots containing a real OTP, public URL you do not want to share, password, secret, or private user information.

### Login
![Login page](screenshots/login.png)

### Signup
![Signup page](screenshots/signup.png)

### Forgot Password — Security Question
![Security question recovery](screenshots/forgot_security_question.png)

### Forgot Password — Email OTP
![OTP recovery page](screenshots/forgot_otp.png)

### OTP Email
![OTP email](screenshots/otp_email.png)

### User Dashboard
![User dashboard](screenshots/user_dashboard.png)

### Admin Dashboard
![Admin dashboard](screenshots/admin_dashboard.png)

> Screenshot files are intentionally not fabricated or pre-filled. Capture them after testing the app and add the image files to this folder.

## 7. Upload to GitHub

1. Create your repository (if needed) and add a `Milestone1` folder.
2. Upload the notebook and `README.md`; upload source `.py` files if your mentor permits.
3. Upload the screenshot files after taking them.
4. **Do not upload** `franchiseops_users.db`, `streamlit.log`, credentials, app passwords, JWT tokens, OTPs, or other secrets.
5. Clear notebook cell outputs before uploading, then inspect every cell and output once more.

## 8. Important security limitations

This is an educational prototype for a controlled demo, not a production authentication service. A public ngrok URL exposes the app to people who have that URL. Use only test data and stop the tunnel after the demo. Streamlit session-state JWT storage is suitable for this milestone but does not implement a hardened production session architecture. Security questions are weaker than email-based or MFA recovery. A production app should use a managed identity provider, persistent server-side sessions, robust rate limiting, audit logging, HTTPS/domain controls, and a managed database.


## Step 9 — Create the database and run quick checks

This cell checks source syntax and core validation/hash functions. It does not create a demo user or print any credentials.


In [ ]:
import py_compile
for source_file in ['database.py', 'auth_utils.py', 'app.py']:
    py_compile.compile(source_file, doraise=True)

from database import init_db
from auth_utils import validate_email, password_issues, hash_password, verify_hash

init_db()
assert validate_email('ab@cd.ef')
assert not validate_email('a@b.c')
assert not password_issues('Abcd1234!')
assert password_issues('weak')
test_hash = hash_password('Abcd1234!')
assert verify_hash('Abcd1234!', test_hash)
assert not verify_hash('not-the-password', test_hash)
print('Syntax check: passed')
print('Validation and password-hash smoke tests: passed')
print('Database initialized successfully.')


## Step 10 — Launch Streamlit and create the ngrok tunnel

Run once, wait for the public URL, and open it in a new browser tab. Do not post the URL publicly. If launch fails, the last part of `streamlit.log` will be shown; confirm dependencies and Colab Secrets first.


In [ ]:
import os
import subprocess
import time
from pathlib import Path
from urllib.request import urlopen
from pyngrok import ngrok

# Stop an app process from a previous run, if there is one.
if 'APP_PROCESS' in globals() and APP_PROCESS.poll() is None:
    APP_PROCESS.terminate()
    try:
        APP_PROCESS.wait(timeout=10)
    except Exception:
        APP_PROCESS.kill()

# Close an old ngrok agent/tunnel if this launch cell was run before.
try:
    ngrok.kill()
except Exception:
    pass

log_path = PROJECT_DIR / 'streamlit.log'
LOG_HANDLE = open(log_path, 'w', encoding='utf-8')
app_env = os.environ.copy()
# The app does not need the ngrok token; keep it out of the child process environment.
app_env.pop('NGROK_AUTHTOKEN', None)

APP_PROCESS = subprocess.Popen(
    ['streamlit', 'run', 'app.py', '--server.port', '8501',
     '--server.address', '0.0.0.0', '--server.headless', 'true'],
    cwd=str(PROJECT_DIR),
    env=app_env,
    stdout=LOG_HANDLE,
    stderr=subprocess.STDOUT,
)

ready = False
for _ in range(40):
    if APP_PROCESS.poll() is not None:
        break
    try:
        with urlopen('http://127.0.0.1:8501/_stcore/health', timeout=2) as response:
            if response.status == 200:
                ready = True
                break
    except Exception:
        time.sleep(1)

if not ready:
    LOG_HANDLE.flush()
    log_text = log_path.read_text(encoding='utf-8', errors='replace')[-5000:]
    raise RuntimeError('Streamlit did not become ready. Recent log output:\n' + log_text)

ngrok.set_auth_token(os.environ['NGROK_AUTHTOKEN'])
TUNNEL = ngrok.connect(addr=8501, proto='http')
print('Streamlit is running. Open this private demo URL in a browser:')
print(TUNNEL.public_url)
print('Stop the app and tunnel with the final notebook cell when finished.')


## Step 11 — Test every required flow

Use a **test account** with an email inbox you control.

1. **Signup:** try valid inputs, an invalid email, a weak password, mismatched passwords, and a duplicate username.
2. **Login:** sign in by username, then log out and test sign-in by email. Bad login should always show one generic error.
3. **Security-question recovery:** look up the test username, answer the saved question, and set a new valid password.
4. **Email OTP recovery:** request an OTP, check the inbox/spam folder, enter the six digits, and reset the password. OTP expires after 10 minutes and is limited to five attempts.
5. **User dashboard:** verify the welcome message and Logout button.
6. **Admin login:** use the credentials configured in Colab Secrets. Confirm the table shows username/email/created time only—no passwords or hashes.

If Gmail does not send mail, verify the account's App Password and `EMAIL_ADDRESS` / `EMAIL_PASSWORD` secrets. Never put the values into code to debug them.


## Step 12 — Stop the public demo


In [ ]:
# Close the public tunnel and stop the Streamlit subprocess.
try:
    ngrok.kill()
except Exception as exc:
    print('ngrok shutdown message:', type(exc).__name__)

if 'APP_PROCESS' in globals() and APP_PROCESS.poll() is None:
    APP_PROCESS.terminate()
    try:
        APP_PROCESS.wait(timeout=10)
    except Exception:
        APP_PROCESS.kill()

try:
    LOG_HANDLE.close()
except Exception:
    pass

print('Streamlit process stopped and ngrok tunnel closed, if they were running.')


## Step 13 — Screenshots, notebook cleanup, and GitHub upload

Capture real screenshots of **Login, Signup, Forgot Password (security question), Forgot Password (OTP), OTP email, User Dashboard, and Admin Dashboard**. Save them in the `screenshots/` directory using the filenames listed in `README.md`.

Before uploading to your Infosys repository:

- Stop the app and tunnel.
- Clear all notebook outputs (**Edit → Clear all outputs**).
- Inspect the notebook and repo for secret values, email App Password, JWT secret, ngrok token, admin password, OTPs, and public URLs.
- Do not upload `franchiseops_users.db`, `streamlit.log`, or any `.env` file.
- Upload the `.ipynb`, `README.md`, the source files if allowed, and screenshots into `Milestone1/`.

**Prototype limits:** the app is appropriate for internship demonstration only. It uses a public ngrok tunnel and SQLite for a lightweight demo; it is not hardened for real customer or employee records.
